# ProductIQ Phase 2 — Step 2.9: Processed Dataset Generation

Persist the 22-column ProductIQ catalog (14 canonical + 8 attribute fields) as a validated Parquet artifact plus manifest.

## 1. Phase objective

Boundary between **data preparation** and **storage/serving**.

Pipeline: raw AJIO CSV → ingestion → validation → cleaning → normalization → deduplication → canonical schema → attribute engineering → **processed dataset artifact**.

Primary outputs (under `resources/processed/`):

- `product_catalog.parquet`
- `product_catalog.manifest.json`

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from data.attributes import ProductAttributeExtractor
from data.catalog import AjioCanonicalProductBuilder
from data.deduplication import AjioDataDeduplicator
from data.export import (
    AJIO_PROCESSED_ROW_COUNT,
    PROCESSED_COLUMN_COUNT,
    PROCESSED_COLUMN_ORDER,
    PROCESSED_PARQUET_FILENAME,
    ProcessedDatasetWriter,
)
from data.ingestion import AjioDataLoader
from data.preprocessing import AjioDataCleaner, AjioDataNormalizer
from data.validation import AjioDataValidator


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_PATH = PROJECT_ROOT / "resources" / "raw" / "Ajio Fashion Clothing.csv"
PROCESSED_DIR = PROJECT_ROOT / "resources" / "processed"
OUTPUT_PATH = PROCESSED_DIR / PROCESSED_PARQUET_FILENAME

raw_df = AjioDataLoader().load(DATASET_PATH)
AjioDataValidator().validate(raw_df)
cleaned_df = AjioDataCleaner().clean(raw_df).dataframe
normalized_df = AjioDataNormalizer().normalize(cleaned_df).dataframe
prepared_df = AjioDataDeduplicator().deduplicate(normalized_df).dataframe
canonical_df = AjioCanonicalProductBuilder().build(prepared_df).dataframe
processed_df = ProductAttributeExtractor().extract(canonical_df).dataframe

print("Processed dataframe rows:", len(processed_df))
print("Processed dataframe columns:", len(processed_df.columns))

## 2. Final 22-column schema

In [ ]:
print("Expected columns:", PROCESSED_COLUMN_COUNT)
print(list(PROCESSED_COLUMN_ORDER))
processed_df.dtypes

## 3. Validate processed dataset contract (pre-write)

In [ ]:
writer = ProcessedDatasetWriter()
pre_write = writer.validate_dataframe(
    processed_df,
    expected_row_count=AJIO_PROCESSED_ROW_COUNT,
)
print(pre_write.summary())

## 4. Write Parquet (atomic) and manifest

In [ ]:
write_result = writer.write(
    processed_df,
    OUTPUT_PATH,
    expected_row_count=AJIO_PROCESSED_ROW_COUNT,
)
print(write_result.report.summary())

## 5. Output metadata

In [ ]:
print(
    {
        "schema_version": write_result.report.schema_version,
        "file_size_bytes": write_result.report.file_size_bytes,
        "checksum": write_result.report.checksum,
        "manifest": write_result.manifest_path.name,
    }
)

## 6. Read Parquet back

In [ ]:
loaded_df = pd.read_parquet(OUTPUT_PATH)
print("Loaded rows:", len(loaded_df))
print("Loaded columns:", len(loaded_df.columns))

## 7. Round-trip integrity

In [ ]:
validation = writer.validate(OUTPUT_PATH, expected_row_count=AJIO_PROCESSED_ROW_COUNT)
print(validation.summary())
print("Column order match:", list(loaded_df.columns) == list(PROCESSED_COLUMN_ORDER))

## 8. Manifest metadata

In [ ]:
manifest = writer.load_manifest(write_result.manifest_path)
manifest

## 9. Checksum verification

In [ ]:
on_disk_checksum = writer.checksum_file(OUTPUT_PATH)
print("Manifest checksum:", manifest["checksum"])
print("On-disk checksum:", on_disk_checksum)
print("Match:", manifest["checksum"] == on_disk_checksum)

## 10. Final artifact summary

- Raw AJIO CSV remains untouched under `resources/raw/`.
- Processed artifact schema version `1.0.0` covers 22 ProductIQ fields.
- Row order follows the preparation pipeline (no shuffle/sort).
- Phase 2.10 (database loading) is out of scope here.